## Preliminaries and Utils

In [1]:
import numpy as np
import pandas as pd
import warnings
import time

from sklearn.cluster import KMeans
from sklearn.preprocessing import  MinMaxScaler, StandardScaler

from modules.prediction_models import OnlineDecisionTreeRegressor
# from modules.prediction_models import OnlineRidgePolynomialRegressor
# from modules.prediction_models import OnlineKNNRegressor

from modules.utils import Metrics, PrintSummary, ShowPlots

In [2]:
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="X has feature names, but PolynomialFeatures was fitted without feature names")
warnings.filterwarnings("ignore", message="X has feature names, but DecisionTreeRegressor was fitted without feature names")
pd.options.mode.chained_assignment = None

In [3]:
summary = PrintSummary()
plots = ShowPlots()
metrics = Metrics()

## Data Loading

In [4]:
data_train = 'datasets/fdata/22_04_train.parquet'
data_test = 'datasets/fdata/22_04_test.parquet'

In [5]:
df_train = pd.read_parquet(data_train, engine="pyarrow").copy()
df_test = pd.read_parquet(data_test, engine="pyarrow").copy()

df_train['Desired QoS'] = df_train['Desired QoS'].astype(int)
df_test['Desired QoS'] = df_test['Desired QoS'].astype(int)

In [6]:
len_test = len(df_test)
len_test

84877

In [7]:
df_test

,Job Number,User ID,Requested Number of Nodes,Requested Number of CPU,Requested Number of GPU,Total Requested Memory,Desired QoS,Requested Time,Run Time,Duration (H4),...,Avg Duration (NH7) 2,Avg Duration (NH7) 3,Avg Duration (NH7) All,Prev Requested Nodes 1,Prev Requested Nodes 2,Prev Requested Nodes 3,Avg Requested Nodes 2,Avg Requested Nodes 3,Avg Requested Nodes All,Submit Time
336833,8687223,1122,82,3936,-1,573046784,127,2400,462,Short,...,1.5,1.333333,1.508870,6.0,82.0,6.0,44.0,31.333333,8.889789,1894043
422694,8773168,299,128,6144,-1,230293504,127,3600,716,Short,...,3.0,3.000000,2.463987,128.0,128.0,128.0,128.0,128.000000,104.140017,1894044
338289,8688742,299,128,6144,-1,233439232,127,3600,1063,Short,...,3.0,3.000000,2.464296,128.0,128.0,128.0,128.0,128.000000,104.153758,1894044
422695,8773169,299,128,6144,-1,228720640,127,3600,721,Short,...,3.0,3.000000,2.464065,128.0,128.0,128.0,128.0,128.000000,104.143454,1894044
422685,8773159,299,128,6144,-1,228130816,127,3600,686,Short,...,3.0,3.000000,2.464219,128.0,128.0,128.0,128.0,128.000000,104.150324,1894044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420451,8771078,1214,1,48,-1,233832448,127,900,658,Short,...,3.0,3.000000,2.176705,1.0,1.0,1.0,1.0,1.000000,25.599521,2432755
420462,8771098,1214,1,48,-1,235864064,127,900,659,Short,...,3.0,3.000000,2.176747,1.0,1.0,1.0,1.0,1.000000,25.598266,2432878
420463,8771099,1214,1,48,-1,233177088,127,900,665,Short,...,3.0,3.000000,2.176789,1.0,1.0,1.0,1.0,1.000000,25.597012,2432894
421342,8771789,1775,4,192,-1,561709056,127,10800,1046,Short,...,4.5,5.000000,3.557692,160.0,4.0,160.0,82.0,108.000000,54.508827,2433016


In [8]:
max_runtime_limit = df_test["Requested Time"].max()
max_runtime_limit

259200

## Data Preparation

In [9]:
scaler = MinMaxScaler()
scaler.fit(df_train['Run Time'].values.reshape(-1, 1))

MinMaxScaler()

In [10]:
fs1 = ["User ID", "Requested Number of Nodes", "Requested Number of CPU", "Requested Number of GPU", "Total Requested Memory", "Desired QoS", "Requested Time"]

fs2 = ["User ID", "Requested Number of Nodes", "Requested Number of CPU", "Requested Number of GPU", "Total Requested Memory", "Desired QoS", "Requested Time",
       "Prev Run Time 1", "Prev Run Time 2", "Prev Run Time 3", "Avg Run Time 2", "Avg Run Time 3", "Avg Run Time All"]

fs3 = ["User ID", "Requested Number of Nodes", "Requested Number of CPU", "Requested Number of GPU", "Total Requested Memory", "Desired QoS", "Requested Time",
       "Prev Run Time 1", "Prev Run Time 2", "Prev Run Time 3", "Avg Run Time 2", "Avg Run Time 3", "Avg Run Time All",
       "Prev Requested Nodes 1", "Prev Requested Nodes 2", "Prev Requested Nodes 3", "Avg Requested Nodes 2", "Avg Requested Nodes 3", "Avg Requested Nodes All"]

target_name = "Run Time"

In [11]:
y_train = df_train[target_name]
y_test  = df_test[target_name]

## SET 1

In [12]:
X_train = df_train[fs1]
X_test  = df_test[fs1]

In [13]:
regressor = OnlineDecisionTreeRegressor(batch_size=50, max_time_limit=max_runtime_limit)
regressor.fit(X_train, y_train)

In [14]:
y_pred = []

start = time.time()
# Simulate streaming job arrivals
for i in range(len(X_test)):
    x = X_test.iloc[[i]]
    y = y_test.iloc[i]
    pred = regressor.predict(x)[0]
    # print("Predicted:", pred, "Actual:", y)
    y_pred.append(pred)
    regressor.partial_fit(x, y)   # online update

end = time.time()

y_pred = np.array(y_pred)

In [15]:
metrics.print(scaler, y_test, y_pred, start, end, len_test)

-------------------------------------------
                 METRICS
-------------------------------------------
Inference time: 83.86610388755798
Latency:        0.000988089869900656
-------------------------------------------
MAE:            3528.892208725568
MAE (hh:mm:ss): 00:58:48
MAE (Scaled):   0.013615393730807333
EA:             0.6727012835181132
MAPE:           5297.198578784748
-------------------------------------------


In [16]:
# Save the results in the file with all other predictions

df = pd.read_csv("results/fdata/predictions.csv")

df["pred_runtime_dt_fs1"] = y_pred

df.to_csv("results/fdata/predictions.csv", index=False)

## SET 2

In [12]:
X_train = df_train[fs2]
X_test  = df_test[fs2]

In [13]:
regressor = OnlineDecisionTreeRegressor(batch_size=50, max_time_limit=max_runtime_limit)
regressor.fit(X_train, y_train)

In [14]:
y_pred = []

start = time.time()
# Simulate streaming job arrivals
for i in range(len(X_test)):
    x = X_test.iloc[[i]]
    y = y_test.iloc[i]
    pred = regressor.predict(x)[0]
    # print("Predicted:", pred, "Actual:", y)
    y_pred.append(pred)
    regressor.partial_fit(x, y)   # online update

end = time.time()

y_pred = np.array(y_pred)

In [15]:
metrics.print(scaler, y_test, y_pred, start, end, len_test)

-------------------------------------------
                 METRICS
-------------------------------------------
Inference time: 104.30247235298157
Latency:        0.0012288661516427486
-------------------------------------------
MAE:            3232.760901068605
MAE (hh:mm:ss): 00:53:52
MAE (Scaled):   0.012472841306055176
EA:             0.6792685964917784
MAPE:           1944.3384355850615
-------------------------------------------


In [16]:
# Save the results in the file with all other predictions

df = pd.read_csv("results/fdata/predictions.csv")

df["pred_runtime_dt"] = y_pred

df.to_csv("results/fdata/predictions.csv", index=False)

## SET 3

In [20]:
X_train = df_train[fs3]
X_test  = df_test[fs3]

In [21]:
regressor = OnlineDecisionTreeRegressor(batch_size=50)
regressor.fit(X_train, y_train)

In [22]:
y_pred = []

start = time.time()
# Simulate streaming job arrivals
for i in range(len(X_test)):
    x = X_test.iloc[[i]]
    y = y_test.iloc[i]
    pred = regressor.predict(x)[0]
    # print("Predicted:", pred, "Actual:", y)
    y_pred.append(pred)
    regressor.partial_fit(x, y)   # online update

end = time.time()

y_pred = np.array(y_pred)

In [23]:
metrics.print(scaler, y_test, y_pred, start, end, len_test)

-------------------------------------------
                 METRICS
-------------------------------------------
Inference time: 95.72128295898438
Latency:        0.0011277646825286517
-------------------------------------------
MAE:            3799.7617611367036
MAE (hh:mm:ss): 01:03:19
MAE (Scaled):   0.014660479663623925
EA:             0.6598973611990688
MAPE:           3639.1061851794666
-------------------------------------------


In [24]:
# Save the results in the file with all other predictions

df = pd.read_csv("results/fdata/predictions.csv")

df["pred_runtime_dt_fs3"] = y_pred

df.to_csv("results/fdata/predictions.csv", index=False)